# Seventh Lab Extension - Linear Actuator - Flying Carpet 

## Import Packages

In [24]:
try
    using Gmsh: gmsh, Gmsh 
catch
    using gmsh
end 

using LinearAlgebra 
using SparseArrays 
using StaticArrays

using BenchmarkTools

using Test 

using Plots
using CairoMakie 
using Meshes

In [25]:
# a point in 2D is a tuple of 2 coordinates 
const Point2D = SVector{2,Float64};

## Section 1: Introduction

To be elaborated. 

## Section 2: Two-Dimensional Mesh Generation

In [26]:
"""
    struct Physics

Holds information for the physical properties of the mesh. 
"""
struct Physics 
  reluctivity::Float64   
  conductivity::Float64  
  currdensity::Float64
end

"""
    struct Element 

Holds information for a single linear triangular element. 
"""
struct Element
  p1::Point2D                   # coordinates first node 
  p2::Point2D                   # coordinates second node 
  p3::Point2D                   # coordinates third node     
  e1::Int64                     # global index first node
  e2::Int64                     # global index second node
  e3::Int64                     # global index third node
  Emat::MMatrix{3,3,Float64, 9} # matrix of basis function coefficients 
  area::Float64                 # area of the element 
  physics::Physics              # physical properties of the element
end

struct Permagn 
    pos::Vector{Int64}
    neg::Vector{Int64} 
end 

"""
    struct Mesh 

Holds information for the entire mesh as an array of linear triangular elements. 
"""
struct Mesh
  nnodes::Int64               # number of nodes 
  nelems::Int64               # number of elements
  dofPerElem::Int64           # number of dofs per element   
  Elements::Array{Element,1}  # list of Elements 
  bndNodeIds::Vector{Int64}   # indices of nodes where Dirichlet bc are applied  
  permagn::Permagn            # permanent magnet 
end

Mesh

In [27]:
"""
    localElement(p1,p2,p3)

Generates area and basis functions for triangular element formed by p1, p2 and p3 
"""
function localElement(p1,p2,p3) 
    p12  = p2 - p1
    p13  = p3 - p1
    area = .5*abs(cross(p12,p13))  
    Xmat = SMatrix{3,3,Float64, 9}(p1[1], p2[1], p3[1], p1[2], p2[2], p3[2], 1, 1, 1) 
    rhs  = SMatrix{3,3,Float64, 9}(1I) 
    Emat = SMatrix{3,3,Float64, 9}(Xmat\rhs);
    return Emat, area
end

localElement

In [262]:
#..sets the source on each group of elements of the mesh 
#..sets the current density in the left (positive) and right (negative) side of the winding
#..the current density is zero on the air and coil domain
#..the source is set to be constant on each element in the mesh 
curr_dens_value = 1e6;
left_coil(group_id)  = (group_id==5)-(group_id==6)
right_coil(group_id) = (group_id==7)-(group_id==8)
sourcefunction(group_id) = curr_dens_value*(left_coil(group_id)+right_coil(group_id))

#..sets the reluctivity on each group of elements of the mesh 
#..sets the reluctivity in the ferromagnetic core
#..the reluctivity is set to one on the coil and air domain 
#..the reluctivity is set to be constant on each element in the mesh 
mu0 = 4*pi*1e-7
mur = 1000; 
function reluctivityfunction(group_id) 
    if (group_id==1)
        return 1/mu0
    elseif (group_id==2) 
        return 1/(mu0*mur)
    elseif (group_id==3) 
        return 1/mu0
    elseif (group_id==4) 
        return 1/(mu0*mur)
    elseif (group_id==5) 
        return 1/mu0
    elseif (group_id==6) 
        return 1/mu0
    elseif (group_id==7) 
        return 1/mu0
    elseif (group_id==8) 
        return 1/mu0 
    else
        println("no group assigned")
    end 
end 


reluctivityfunction (generic function with 1 method)

## Section 2: Read Mesh From File and Store Mesh in Struct

In [263]:
# group elements in coil, core and air domain 
function groupElements!(e_group,gmshModel)

    #..Get elements from the mesh (more)
    element_types, element_ids, element_connectivity = gmshModel.mesh.getElements(2)
    nelems = length(element_ids[1])
     
    #..Get nodes from physical subdomains (more)
    ngroup1 = gmsh.model.mesh.getNodesForPhysicalGroup(2, 1)
    ngroup2 = gmsh.model.mesh.getNodesForPhysicalGroup(2, 2)
    ngroup3 = gmsh.model.mesh.getNodesForPhysicalGroup(2, 3)
    ngroup4 = gmsh.model.mesh.getNodesForPhysicalGroup(2, 4)
    ngroup5 = gmsh.model.mesh.getNodesForPhysicalGroup(2, 5)
    ngroup6 = gmsh.model.mesh.getNodesForPhysicalGroup(2, 6) 
    ngroup7 = gmsh.model.mesh.getNodesForPhysicalGroup(2, 7)
    ngroup8 = gmsh.model.mesh.getNodesForPhysicalGroup(2, 8)

    #..Set group number on each element
    for element_id in 1:nelems
        node1_id = element_connectivity[1][3*(element_id-1)+1]
        node2_id = element_connectivity[1][3*(element_id-1)+2]
        node3_id = element_connectivity[1][3*(element_id-1)+3]
        G1 = sum(node1_id.== ngroup1[1])+sum(node2_id.== ngroup1[1])+sum(node3_id.== ngroup1[1]) #air 
        G2 = sum(node1_id.== ngroup2[1])+sum(node2_id.== ngroup2[1])+sum(node3_id.== ngroup2[1]) #core
        G3 = sum(node1_id.== ngroup3[1])+sum(node2_id.== ngroup3[1])+sum(node3_id.== ngroup3[1]) #mover
        G4 = sum(node1_id.== ngroup4[1])+sum(node2_id.== ngroup4[1])+sum(node3_id.== ngroup4[1]) #magnet
        G5 = sum(node1_id.== ngroup5[1])+sum(node2_id.== ngroup5[1])+sum(node3_id.== ngroup5[1]) #left_coil_up
        G6 = sum(node1_id.== ngroup6[1])+sum(node2_id.== ngroup6[1])+sum(node3_id.== ngroup6[1]) #left_coil_dw
        G7 = sum(node1_id.== ngroup7[1])+sum(node2_id.== ngroup7[1])+sum(node3_id.== ngroup7[1]) #right_coil_up
        G8 = sum(node1_id.== ngroup8[1])+sum(node2_id.== ngroup8[1])+sum(node3_id.== ngroup8[1]) #right_coil_dw
        if G1 == 3
           e_group[element_id] = 1;
        elseif G2 == 3 
           e_group[element_id] = 2;
        elseif G3 == 3
           e_group[element_id] = 3;
        elseif G4 == 3
           e_group[element_id] = 4;
        elseif G5 == 3
           e_group[element_id] = 5;
        elseif G6 == 3
           e_group[element_id] = 6;
        elseif G7 == 3
           e_group[element_id] = 7;
        elseif G8 == 3
           e_group[element_id] = 8; 
        else 
            println("no group assignded to elememt ", element_id)
        end
        
        if (false)
           println("on element ", element_id, " e_group[element_id] = ", e_group[element_id])
        end
     end 
     return e_group 
end

groupElements! (generic function with 1 method)

In [369]:
"""
    meshFromGmsh(meshFile)

Reads mesh from an external file and stores mesh in data structure 
"""
function meshFromGmsh(meshFile)    
    
    #..Initialize GMSH
    gmsh.initialize()
    
    #..Read mesh from file
    gmsh.open(meshFile)

    #..Get the mesh nodes
    node_ids, node_coord, _ = gmsh.model.mesh.getNodes()
    nnodes = length(node_ids)
    xnode_coord = node_coord[1:3:end]; ynode_coord = node_coord[2:3:end]
    
    #..Get the mesh elements
    #..Observe that we get all the two-dimensional triangular elements from the mesh
    element_types, element_ids, element_connectivity = gmsh.model.mesh.getElements(2)
    nelems = length(element_ids[1])

    #..Get the physical groups 
    e_group = zeros(nelems)
    e_group = groupElements!(e_group,gmsh.model)
    reluctivityperelement = map(reluctivityfunction, e_group)    
    sourceperelement = map(sourcefunction, e_group)

    #..Construct uninitialized array of length nelements  
    Elements = Array{Element}(undef,nelems)

    #..Construct the array of elements 
    for element_id in 1:nelems
        e1 = element_connectivity[1][3*(element_id-1)+1]
        e2 = element_connectivity[1][3*(element_id-1)+2]
        e3 = element_connectivity[1][3*(element_id-1)+3]
        xnode1 = xnode_coord[e1]; ynode1 = ynode_coord[e1]
        xnode2 = xnode_coord[e2]; ynode2 = ynode_coord[e2]
        xnode3 = xnode_coord[e3]; ynode3 = ynode_coord[e3]
        p1 = Point2D(xnode1,ynode1) 
        p2 = Point2D(xnode2,ynode2) 
        p3 = Point2D(xnode3,ynode3)
        Emat, area = localElement(p1,p2,p3) 
        physics = Physics(reluctivityperelement[element_id], 1., sourceperelement[element_id])
        Elements[element_id] = Element(p1,p2,p3,e1,e2,e3,Emat,area,physics)
    end

    #..retrieve boundary nodes by loop over corner point and boundary edges
    node_ids1=[]; node_ids2=[]; node_ids3=[]; node_ids4=[]; 
    node_ids5=[]; node_ids6=[]; node_ids7=[]; node_ids8=[]; 
    node_ids1, node_coord, _ = gmsh.model.mesh.getNodes(0,197)
    node_ids2, node_coord, _ = gmsh.model.mesh.getNodes(0,198)
    node_ids3, node_coord, _ = gmsh.model.mesh.getNodes(0,199)
    node_ids4, node_coord, _ = gmsh.model.mesh.getNodes(0,200)
    node_ids5, node_coord, _ = gmsh.model.mesh.getNodes(1,203)
    node_ids6, node_coord, _ = gmsh.model.mesh.getNodes(1,204)
    node_ids7, node_coord, _ = gmsh.model.mesh.getNodes(1,205)
    node_ids8, node_coord, _ = gmsh.model.mesh.getNodes(1,206)
    bnd_node_ids = union(node_ids1,node_ids2,node_ids3,node_ids4,node_ids5,node_ids6,node_ids7,node_ids8)

    #..retrieve boundary nodes by loop over corner point and boundary edges
    node_ids1=[]; node_ids2=[]; node_ids3=[]; node_ids4=[];
    node_ids1, node_coord, _ = gmsh.model.mesh.getNodes(1,90)
    node_ids2, node_coord, _ = gmsh.model.mesh.getNodes(1,88)
    permagn = Permagn(node_ids1,node_ids2) 
        
    #..Set DOF per element
    dofPerElement = 3
    
    #..Store data inside mesh struct  
    mesh = Mesh(nnodes,nelems,dofPerElement,Elements,bnd_node_ids,permagn) 
    
    #..Finalize gmsh
    gmsh.finalize()
    
    return mesh, xnode_coord, ynode_coord
end

meshFromGmsh

In [371]:
mymesh, xnode, ynode = meshFromGmsh("linear-actuator.msh");

Info    : Reading 'linear-actuator.msh'...
Info    : 144 entities
Info    : 67831 nodes
Info    : 135660 elements
Info    : Done reading 'linear-actuator.msh'                                                                     


## Section 3: FEM Assembly 

### Section 1.3: Stiffness Matrix Assembly

In [372]:
"""
    genLocStiffMat(element::Element)

Generates local stiffness matrix for single linear triangular element. 
"""
function genLocStiffMat(element::Element)
    v    = SVector(element.e1, element.e2, element.e3)   
    Emat = copy(element.Emat) 
    area = element.area 
    reluctivity = element.physics.reluctivity
    Iloc = SVector{9}(v[i] for j=1:3, i=1:3)
    Jloc = SVector{9}(v[i] for i=1:3, j=1:3) 
    Emat[3,:] .= 0.;  
    Amat = SMatrix{3,3}(area*reluctivity*(transpose(Emat)*Emat))
    Aloc = vec(Amat)
    return Iloc, Jloc, Aloc
end

"""
    genStiffMat(mesh::Mesh)

Generates global stiffness matrix on mesh of linear triangular elements. 
"""
function genStiffMat(mesh::Mesh)
 
    #..recover number of elements  
    nnodes      = mesh.nnodes 
    nelems      = mesh.nelems 
    dofPerElem  = mesh.dofPerElem
    dofPerElem2 = dofPerElem^2

    #..set range vector 
    irng = SVector{9}(1:9)

    #..preallocate the memory for local matrix contributions 
    Avalues = zeros(Float64,dofPerElem2*nelems)
    I = zeros(Int64,length(Avalues))
    J = zeros(Int64,length(Avalues))

    #..loop over number of elements..
    for element in mesh.Elements 
        Iloc, Jloc, Aloc = genLocStiffMat(element) 
        I[irng] .= Iloc 
        J[irng] .= Jloc 
        Avalues[irng] .= Aloc   
        irng = irng.+dofPerElem2
    end
 
    #..form sparse matrix        
    A = sparse(I,J,Avalues,nnodes,nnodes)
   
    return A; 
end

genStiffMat

In [373]:
A = genStiffMat(mymesh);

### Section 2.3: Mass Matrix Assembly

In [374]:
"""
    genLocMassMat(element::Element)

Generates local mass matrix for single linear triangular element. 
"""
function genLocMassMat(element::Element)
    v    = SVector(element.e1, element.e2, element.e3)  
    Emat = copy(element.Emat)
    area = element.area 
    conductivity = element.physics.conductivity
    Iloc = SVector{9}(v[i] for j=1:3, i=1:3)
    Jloc = SVector{9}(v[i] for i=1:3, j=1:3) 
    Mmat = area/3*conductivity*SMatrix{3,3}(1I) 
    Mloc = vec(Mmat)
    return Iloc, Jloc, Mloc
end

"""
    genMassMat(mesh::Mesh)

Generates global mass matrix on mesh of linear triangular elements. 
"""
function genMassMat(mesh::Mesh)
 
    #..recover number of elements  
    nnodes      = mesh.nnodes 
    nelems      = mesh.nelems 
    dofPerElem  = mesh.dofPerElem
    dofPerElem2 = dofPerElem^2

    #..set range vector 
    irng = SVector{9}(1:9)

    #..preallocate the memory for local matrix contributions 
    Mvalues = zeros(Float64,dofPerElem2*nelems)
    I = zeros(Int64,length(Mvalues))
    J = zeros(Int64,length(Mvalues)) 
    
    #..loop over number of elements..
    for element in mesh.Elements 
        Iloc, Jloc, Mloc = genLocMassMat(element) 
        I[irng] .= Iloc 
        J[irng] .= Jloc 
        Mvalues[irng] .= Mloc   
        irng = irng.+dofPerElem2
    end

    #..form sparse matrix
    M = sparse(I,J,Mvalues)
   
    return M; 
end

genMassMat

In [375]:
M = genMassMat(mymesh);

### Section 3.3: Right-Hand Side Assembly

In [389]:
"""
    genLocVector(element::Element, sourceFct::Function)

Generates local load vector for single linear triangular element. 
""" 
function genLocVector(element::Element)
    ploc = SVector(element.p1, element.p2, element.p3)   
    Iloc = SVector(element.e1, element.e2, element.e3)   
    area = element.area 
    currdensity = element.physics.currdensity
    floc = area/3*currdensity*SVector(1.,1.,1.)
    return Iloc, floc
end

"""
    genVector(mesh, sourceFct::F) where F

Generates global load vector on mesh of linear triangular elements. 
"""
function genVector(mesh) 
 
    #..recover number of nodes  
    nnodes = mesh.nnodes 
     
    #..preallocate the memory for local matrix contributions 
    f = zeros(Float64,nnodes)

    #..loop over number of elements..
    for element in mesh.Elements
        Iloc, floc = genLocVector(element)
        f[Iloc] += floc 
    end

    f[mesh.permagn.pos] .= 10.
    f[mesh.permagn.neg] .= -10.

    return f; 
end

genVector

In [390]:
f = genVector(mymesh);

## Section 4: FEM Solve  

### Section 1.4: Handle Dirichlet Essential Boundary Conditions 

In [391]:
"""
    handleBoundary!(mesh,A,f)

Handles the Dirichlet boundary conditions  
"""
function handleBoundary!(mesh,A,f)
    bndNodeIds = mesh.bndNodeIds; 
    #..handle essential boundary conditions 
    A[bndNodeIds,:] .= 0;
    A[bndNodeIds,bndNodeIds] = Diagonal(ones(size(bndNodeIds)))
    f[bndNodeIds] .= 0;
    return A, f  
end

handleBoundary!

In [392]:
A,f = handleBoundary!(mymesh,A,f);

### Section 2.4: Solve Linear System 

In [393]:
"""
    genSolution(mesh,A,f)

Generates the finite element solution 
"""
function genSolution!(mesh,A,f)
    A, f = handleBoundary!(mesh,A,f)
    u = A\f 
    return u 
end

genSolution!

In [394]:
u = genSolution!(mymesh,A,f);

## Section 4: Post-Processing for the Magnetic Flux 

In [395]:
"""
    genDerivLoc(element::Element,u)

Generates local flux vector for single linear triangular element.
"""
function genDerivLoc(element,u)
    ploc = SVector(element.p1, element.p2, element.p3)   
    Iloc = SVector(element.e1, element.e2, element.e3)   
    Emat = copy(element.Emat)
    Emat[3,:] .= 0.; 
    area = element.area 
    uloc = u[Iloc] 
    xmid = sum(ploc)/3
    Bx = -dot(uloc,Emat[2,:])
    By = dot(uloc,Emat[1,:])
    normB2 = Bx^2 + By^2 
    return xmid, Bx, By, normB2
end

"""
    genDeriv(mesh, u)

Generates global flux vector on mesh of linear triangular elements. 
"""
function genDeriv(mesh, u)

    #..recover number of elements  
    nelems = mesh.nelems 
    nnodes = mesh.nnodes 

    #..allocate memory for arrays 
    xmid   = zeros(Point2D,nelems)
    Bx     = zeros(Float64,nelems)
    By     = zeros(Float64,nelems)
    normB2 = zeros(Float64,nelems)
    Hx     = zeros(Float64,nelems)
    Hy     = zeros(Float64,nelems)
    normH2 = zeros(Float64,nelems)
    Wm     = zeros(Float64,nelems)
    
    #..loop over elements to compute derivative information 
    for (k,element) in enumerate(mesh.Elements) 
        xmid[k], Bx[k], By[k], normB2[k] = genDerivLoc(element,u)
        Hx[k]     = Bx[k]*element.physics.reluctivity; Hy[k] = By[k]*element.physics.reluctivity
        normH2[k] = Hx[k]^2 + Hy[k]^2
        Wm[k]     = Bx[k]*Hx[k] + By[k]*Hy[k]
    end 
    
    return xmid, Bx, By, normB2, Hx, Hy, normH2, Wm 
end

genDeriv

In [396]:
xmid, Bx, By, normB2, Hx, Hy, normH2, Wm = genDeriv(mymesh, u);

## Section 7: Write solution to VTK file 

In [397]:
using WriteVTK

In [398]:
points = [xnode ynode]'
cells = [MeshCell(VTKCellTypes.VTK_TRIANGLE, [element.e1, element.e2, element.e3]) for element in mymesh.Elements]

vtk_grid("linear-actuator", points, cells) do vtk
    vtk["potential",VTKPointData()] = u
    vtk["x-magnflux",VTKCellData()] = Bx 
    vtk["y-magnflux",VTKCellData()] = By
    vtk["mag-magnflux",VTKCellData()] = normB2 
    vtk["x-magnfield",VTKCellData()] = Hx 
    vtk["y-magnfield",VTKCellData()] = Hy
    vtk["mag-magnfield",VTKCellData()] = normH2    
    vtk["mag-energy",VTKCellData()] = Wm 
    vtk_save(vtk) 
end

1-element Vector{String}:
 "computed_solution.vtu"